In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import time

STORAGE_ACCOUNT = "adlssupplychain"
CONTAINER       = "supplychain-data"

STORAGE_KEY = dbutils.secrets.get(scope="adls-scope", key="adls-account-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)


BASE_PATH    = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
SILVER_DELTA = f"{BASE_PATH}silver/supply_chain_100gb"

In [0]:
# ── 2. OPTIMIZE + Z-ORDER ──────────────────────────────────────

print(f"Starting OPTIMIZE + Z-ORDER on: {SILVER_DELTA}")
print("Tuning data layout for distributed ML performance...")

t0 = time.time()

spark.sql("""
    OPTIMIZE supply_chain_db.supply_chain_100gb
    ZORDER BY (
        `Late_delivery_risk`,
        `Order_Country`
    )
""")

elapsed = time.time() - t0
print(f"OPTIMIZE + Z-ORDER complete: {elapsed:.0f}s ({elapsed/60:.1f} min)")

Starting OPTIMIZE + Z-ORDER on: abfss://supplychain-data@adlssupplychain.dfs.core.windows.net/silver/supply_chain_100gb
Tuning data layout for distributed ML performance...
OPTIMIZE + Z-ORDER complete: 830s (13.8 min)


In [0]:
#  Post-optimize stats 
spark.sql(f"DESCRIBE DETAIL delta.`{SILVER_DELTA}`") \
     .select("numFiles", "sizeInBytes", "partitionColumns") \
     .show(truncate=False)

#  Vacuum: keep 7-day history, remove old files 
print("Running VACUUM (retaining 168 hours for Time Travel)...")
spark.sql(f"VACUUM delta.`{SILVER_DELTA}` RETAIN 168 HOURS")
print("VACUUM complete.")

#  Partition distribution check 
print("\nPartition distribution by Market:")
spark.table("supply_chain_db.supply_chain_100gb") \
     .groupBy("Market") \
     .count() \
     .orderBy(F.desc("count")) \
     .show()

print("\nPartition distribution by Order Region (top 15):")
spark.table("supply_chain_db.supply_chain_100gb") \
     .groupBy("Order_Region") \
     .count() \
     .orderBy(F.desc("count")) \
     .show(15)

+--------+-----------+----------------------+
|numFiles|sizeInBytes|partitionColumns      |
+--------+-----------+----------------------+
|59      |17103766914|[Market, Order_Region]|
+--------+-----------+----------------------+

Running VACUUM (retaining 168 hours for Time Travel)...
VACUUM complete.

Partition distribution by Market:
+------------+--------+
|      Market|   count|
+------------+--------+
|       LATAM|29081111|
|      Europe|28355461|
|Pacific Asia|23287246|
|        USCA|14555868|
|      Africa| 6560767|
+------------+--------+


Partition distribution by Order Region (top 15):
+---------------+--------+
|   Order_Region|   count|
+---------------+--------+
|Central America|16003191|
| Western Europe|15289426|
|  South America| 8411872|
|        Oceania| 5707261|
|Northern Europe| 5549507|
| Southeast Asia| 5415815|
|Southern Europe| 5285035|
|      Caribbean| 4666048|
|   West of USA | 4495009|
|     South Asia| 4362588|
|   Eastern Asia| 4084517|
|    East of USA